# CBIA - Construction Bid Intelligence Assistant

Asistente de IA que analiza Bid Packages de construcción (PDFs), los clasifica,
los indexa con embeddings locales y responde preguntas usando RAG
(Retrieval-Augmented Generation).

**Flujo del notebook:**
1. Instalar dependencias
2. Configurar la API key (Colab Secrets)
3. Subir y extraer el Bid Package
4. Clasificar documentos
5. Cargar PDFs y añadir metadata
6. Dividir en chunks
7. Generar embeddings y construir la base vectorial (FAISS)
8. Cargar el system prompt
9. Crear el LLM (OpenRouter) y preguntar

---

### 🔑 Configuración de la API key (una sola vez)
1. Click en el ícono de llave 🔑 en la barra lateral izquierda de Colab.
2. Crea un secreto nuevo llamado exactamente `OPENROUTER_API_KEY`.
3. Pega tu key como valor y activa **"Notebook access"**.
4. Nunca escribas la key directamente en una celda.

## 1. Instalación de dependencias

In [1]:
!pip install -q langchain langchain-community langchain-openai langchain-text-splitters \
    pypdf faiss-cpu sentence-transformers gradio

print("✅ CBIA environment ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
✅ CBIA environment ready!


In [2]:
import os
import zipfile
from collections import Counter

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI

print("✅ Librerías importadas")

/tmp/ipykernel_1355/3149886180.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ Librerías importadas


## 2. Configuración del entorno (API key)

Obtiene la API key de OpenRouter de forma segura, sin escribirla nunca en el código.
Busca primero en **Colab Secrets** y, si no está disponible (por ejemplo fuera de Colab),
en un archivo `.env`.

In [3]:
def setup_environment() -> str:
    """
    Obtiene la API key de OpenRouter de forma segura.

    Orden de búsqueda:
        1. Colab Secrets (google.colab.userdata) -> forma recomendada en Colab
        2. Variable de entorno / archivo .env -> forma recomendada fuera de Colab
    """
    api_key = None

    # --- Opción 1: Google Colab Secrets ---
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENROUTER_API_KEY")
        if api_key:
            print("✅ API key cargada desde Colab Secrets")
    except ImportError:
        pass
    except Exception:
        pass

    # --- Opción 2: archivo .env / variables de entorno (fuera de Colab) ---
    if not api_key:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except ImportError:
            pass
        api_key = os.getenv("OPENROUTER_API_KEY")
        if api_key:
            print("✅ API key cargada desde variables de entorno (.env)")

    if not api_key:
        raise ValueError(
            "No se encontró OPENROUTER_API_KEY.\n\n"
            "En Colab: abre el ícono de llave 🔑 en la barra lateral, crea "
            "un secreto llamado OPENROUTER_API_KEY, pega tu key y activa "
            "'Notebook access'.\n\n"
            "Fuera de Colab: crea un archivo .env con:\n"
            "OPENROUTER_API_KEY=tu_clave_aqui"
        )

    return api_key


api_key = setup_environment()

✅ API key cargada desde Colab Secrets


## 3. Carga y extracción del Bid Package

In [4]:
def upload_bid_package_colab() -> str:
    """Abre el diálogo de subida de archivos de Colab y devuelve la ruta del .zip subido."""
    from google.colab import files
    uploaded = files.upload()
    zip_path = list(uploaded.keys())[0]
    print(f"📤 Archivo subido: {zip_path}")
    return zip_path


def extract_bid_package(zip_path: str) -> str:
    """Extrae un Bid Package en formato .zip a una carpeta local con el mismo nombre."""
    extract_folder = os.path.splitext(zip_path)[0]
    os.makedirs(extract_folder, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_folder)

    print("===================================")
    print("📦 Bid Package cargado correctamente")
    print(f"Archivo: {zip_path}")
    print(f"Carpeta de trabajo: {extract_folder}")
    print("===================================")

    return extract_folder

In [5]:
# Sube tu Bid Package en formato .zip
zip_path = upload_bid_package_colab()
extract_folder = extract_bid_package(zip_path)

Saving Bid_Package.zip to Bid_Package.zip
📤 Archivo subido: Bid_Package.zip
📦 Bid Package cargado correctamente
Archivo: Bid_Package.zip
Carpeta de trabajo: Bid_Package


## 4. Clasificación de documentos

In [6]:
def classify_document(filename: str) -> str:
    """Clasifica un documento del Bid Package según palabras clave en su nombre."""
    name = filename.lower()

    if "bond" in name:
        return "Bond Forms"
    elif "addendum" in name:
        return "Addendum"
    elif "plan" in name or "specification" in name:
        return "Plans and Specifications"
    elif "condition" in name:
        return "General Conditions"
    elif "agreement" in name or "contract" in name:
        return "Contract Documents"
    elif "itb" in name or "invitation" in name:
        return "Bid Instructions"
    else:
        return "Other Documents"


def build_document_inventory(extract_folder: str) -> list[dict]:
    """Recorre la carpeta extraída y arma un inventario de documentos con su categoría."""
    inventory = []
    for root, _, files in os.walk(extract_folder):
        for file in sorted(files):
            inventory.append({"filename": file, "category": classify_document(file)})
    return inventory


def print_inventory_summary(inventory: list[dict], extract_folder: str) -> None:
    """Imprime un resumen legible del inventario de documentos."""
    print("\n📋 BID PACKAGE INVENTORY\n")
    for doc in inventory:
        print(f"📄 {doc['filename']}")
        print(f"   Categoría: {doc['category']}\n")

    category_summary = Counter(doc["category"] for doc in inventory)

    print("==========================================")
    print("🏗️ CBIA - Bid Package Assessment")
    print("==========================================\n")
    print(f"📦 Package: {extract_folder}")
    print(f"📄 Total documents: {len(inventory)}\n")
    print("Document Categories:\n")

    for category, count in category_summary.items():
        print(f"✔ {category}: {count}")

    print("\n------------------------------------------")
    print("Status: ✅ READY FOR ANALYSIS")
    print("------------------------------------------")

In [7]:
inventory = build_document_inventory(extract_folder)
print_inventory_summary(inventory, extract_folder)


📋 BID PACKAGE INVENTORY

📄 ADDENDUM_2_-_Fuel_Bay_Overhead_Canopy_Replacement.pdf
   Categoría: Addendum

📄 ADDENDUM_3_-_Fuel_Bay_Overhead_Canopy_Replacement.pdf
   Categoría: Addendum

📄 Bid_Bond form.pdf
   Categoría: Bond Forms

📄 Construction_Plans_and_Specifications.pdf
   Categoría: Plans and Specifications

📄 ITB26-6310-25B_-_Fuel_Bay_Overhead_Canopy_Replacement.pdf
   Categoría: Bid Instructions

📄 PAYMENT_BOND_.pdf
   Categoría: Bond Forms

📄 Performance_Bond.pdf
   Categoría: Bond Forms

📄 Sample_Construction_Agreement.pdf
   Categoría: Contract Documents

📄 Standard_General_Conditions_of_the_Construction_Contract.pdf
   Categoría: General Conditions

🏗️ CBIA - Bid Package Assessment

📦 Package: Bid_Package
📄 Total documents: 9

Document Categories:

✔ Addendum: 2
✔ Bond Forms: 3
✔ Plans and Specifications: 1
✔ Bid Instructions: 1
✔ Contract Documents: 1
✔ General Conditions: 1

------------------------------------------
Status: ✅ READY FOR ANALYSIS
--------------------------

## 5. Carga de PDFs y enriquecimiento con metadata

In [8]:
def load_pdf_documents(extract_folder: str) -> list:
    """Carga el contenido de todos los PDFs encontrados en la carpeta."""
    documents = []
    for root, _, files in os.walk(extract_folder):
        for file in sorted(files):
            if file.lower().endswith(".pdf"):
                file_path = os.path.join(root, file)
                loader = PyPDFLoader(file_path)
                documents.extend(loader.load())

    print("===================================")
    print("📚 CBIA Document Loading Completed")
    print("===================================")
    print(f"Total pages loaded: {len(documents)}")
    return documents


def enhance_documents_with_metadata(documents: list, inventory: list[dict]) -> list:
    """Añade a cada página su categoría y normaliza el número de página (base 1)."""
    category_map = {doc["filename"]: doc["category"] for doc in inventory}

    for doc in documents:
        filename = os.path.basename(doc.metadata["source"])
        doc.metadata["category"] = category_map.get(filename, "Unknown")
        doc.metadata["page_number"] = doc.metadata["page"] + 1

    print("===================================")
    print("📚 CBIA Enhanced Documents Created")
    print("===================================")
    print(f"Total pages processed: {len(documents)}")
    return documents

In [9]:
documents = load_pdf_documents(extract_folder)
documents = enhance_documents_with_metadata(documents, inventory)

📚 CBIA Document Loading Completed
Total pages loaded: 112
📚 CBIA Enhanced Documents Created
Total pages processed: 112


## 6. División en chunks

In [10]:
def split_into_chunks(documents: list, chunk_size: int = 1000, chunk_overlap: int = 200) -> list:
    """Divide los documentos en fragmentos ("chunks") para poder indexarlos."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(documents)

    print("===================================")
    print("🧠 CBIA Knowledge Chunks Created")
    print("===================================")
    print(f"Original pages   : {len(documents)}")
    print(f"Knowledge chunks : {len(chunks)}")
    return chunks


chunks = split_into_chunks(documents)

🧠 CBIA Knowledge Chunks Created
Original pages   : 112
Knowledge chunks : 505


## 7. Embeddings y base vectorial (FAISS)

Se usan embeddings **locales** (HuggingFace) en vez de los de Gemini, porque el
plan gratuito de Gemini tiene límites de cuota que interrumpían la indexación.

In [11]:
def build_vectorstore(chunks: list) -> FAISS:
    """Genera embeddings locales y construye la base vectorial FAISS."""
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    print("===================================")
    print("🧠 Building CBIA Knowledge Base...")
    print("===================================")

    vectorstore = FAISS.from_documents(chunks, embeddings)

    print("✅ CBIA Knowledge Base Created")
    print(f"Indexed chunks: {len(chunks)}")
    return vectorstore


vectorstore = build_vectorstore(chunks)

/tmp/ipykernel_1355/504452784.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🧠 Building CBIA Knowledge Base...
✅ CBIA Knowledge Base Created
Indexed chunks: 505


## 8. System prompt

In [12]:
DEFAULT_SYSTEM_PROMPT = """
Eres CBIA (Construction Bid Intelligence Assistant).

Tu misión es ayudar a estimadores y contratistas a analizar Bid Packages de construcción.

REGLAS:
- Usa únicamente la información encontrada en el Bid Package.
- Nunca inventes información.
- Siempre cita el documento utilizado.
- Siempre cita la categoría.
- Siempre cita la página.
- Si la información no existe, dilo claramente.
- Si hay conflicto entre documentos, indícalo.
- Los Addenda prevalecen sobre documentos anteriores.
- Organiza siempre la información de forma clara.
- Reduce el tiempo que el estimador dedica a buscar información.

Formato de respuesta:
Respuesta
Evidencia
Documento
Categoría
Página
Notas adicionales
"""


def load_system_prompt(path: str = "system_prompt.md") -> str:
    """Carga el system prompt desde un archivo .md si existe; si no, usa el prompt por defecto."""
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            prompt = f.read()
        print(f"✅ System Prompt cargado desde {path}")
        return prompt

    print("⚠️ No se encontró system_prompt.md, usando prompt por defecto")
    return DEFAULT_SYSTEM_PROMPT


system_prompt = load_system_prompt()

⚠️ No se encontró system_prompt.md, usando prompt por defecto


## 9. LLM (OpenRouter) y motor de respuestas

In [13]:
def build_llm(api_key: str, model: str = "cohere/north-mini-code:free") -> ChatOpenAI:
    """Crea el cliente del LLM apuntando a OpenRouter."""
    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
        model=model,
        temperature=0.1,
    )
    print("✅ OpenRouter conectado")
    return llm


def retrieve_context(vectorstore: FAISS, question: str, k: int = 5) -> str:
    """Busca los fragmentos más relevantes y los formatea como contexto."""
    docs = vectorstore.similarity_search(question, k=k)

    context_parts = []
    for doc in docs:
        context_parts.append(
            f"\nDOCUMENT: {doc.metadata['source'].split('/')[-1]}\n"
            f"CATEGORY: {doc.metadata['category']}\n"
            f"PAGE: {doc.metadata['page_number']}\n\n"
            f"{doc.page_content}\n"
            "------------------------------------------------------\n"
        )
    return "".join(context_parts)


def ask_cbia(question: str, vectorstore: FAISS, llm: ChatOpenAI, system_prompt: str) -> str:
    """Recupera contexto relevante del Bid Package y le pide al LLM una respuesta estructurada."""
    context = retrieve_context(vectorstore, question)

    prompt = f"""
{system_prompt}

You are analyzing a construction bid package.

Use ONLY the following evidence:

{context}

USER QUESTION:
{question}

Provide a structured answer with:
1. Answer
2. Evidence used
3. Source documents
4. Pages
5. Confidence level
"""

    response = llm.invoke(prompt)
    return response.content


llm = build_llm(api_key)

✅ OpenRouter conectado


## 10. Preguntar a CBIA

In [15]:
question = "What are the required documents for bid submission?"
answer = ask_cbia(question, vectorstore, llm, system_prompt)

print("===================================")
print("💬 Respuesta de CBIA")
print("===================================\n")
print(answer)

💬 Respuesta de CBIA

**Respuesta**
El licitador debe presentar los siguientes documentos / anexos obligatorios para completar la presentación de la oferta:

1. **Anexo de experiencia de personas clave** – nombre y licencia del individuo que tendrá supervisión personal del trabajo.
2. **Documentación de equipos** – (a) Equipos de propiedad disponible para el trabajo, (b) Equipos que se comprarán para el trabajo propuesto, (c) Equipos que se alquilarán para el trabajo propuesto.
3. **Certificado de estatus, competencia y / o registro estatal** del licitador.
4. **Formulario de confirmación** (la declaración de que el licitador reconoce que la información será utilizada por la ciudad y que está garantizada como verdadera).
5. **Formulario de verificación de antecedentes penales** (el formulario de verificación de antecedentes penales es el requisito de verificación de antecedentes referenced en los documentos de la licitación).

---

**Evidencia utilizada**

| # | Evidencia (extracto) | D

In [16]:
# Cambia la pregunta y vuelve a ejecutar esta celda para hacer más consultas
question = "Escribe aquí tu pregunta sobre el bid package"
answer = ask_cbia(question, vectorstore, llm, system_prompt)
print(answer)

APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}}

In [14]:
response = llm.invoke("Say hello and confirm you are working.")
print(response.content)

Hello! Yes, I'm up and running—ready to help you out. How can I assist you today?
